In [ ]:
import pandas as pd

In [ ]:
%sql
DROP DATABASE IF EXISTS workspace.silver_weather CASCADE;

In [ ]:
%sql
CREATE DATABASE IF NOT EXISTS workspace.silver_weather
COMMENT 'Capa Silver: clima tipado y limpio' 

In [ ]:
HOURLY_TYPES = {
    "location_id": "int64",
    "temperature_2m": "float64",
    "relative_humidity_2m": "float64",
    "precipitation": "float64",
    "wind_speed_10m": "float64",
    "wind_gusts_10m": "float64",
    "wind_direction_10m": "float64",
    "shortwave_radiation": "float64",
    "et0_fao_evapotranspiration": "float64",
    "uv_index": "float64",
    "weather_code": "int64",
}

hourly = spark.table("workspace.bronze_weather.weather_hourly").toPandas()
hourly = hourly.rename(columns={"time": "observation_time"})
hourly["observation_time"] = pd.to_datetime(hourly["observation_time"])
hourly = hourly.astype(HOURLY_TYPES)
hourly = hourly.drop_duplicates(subset=["location_id", "observation_time", "data_type"])

df_spark = spark.createDataFrame(hourly)
df_spark.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver_weather.weather_hourly")

In [ ]:
DAILY_TYPES = {
    "location_id": "int64",
    "weather_code": "int64",
    "temperature_2m_max": "float64",
    "temperature_2m_min": "float64",
    "temperature_2m_mean": "float64",
    "precipitation_sum": "float64",
    "wind_speed_10m_max": "float64",
    "wind_gusts_10m_max": "float64",
    "shortwave_radiation_sum": "float64",
    "et0_fao_evapotranspiration": "float64",
    "uv_index_max": "float64",
    "sunshine_duration": "float64",
}

daily = spark.table("workspace.bronze_weather.weather_daily").toPandas()
daily = daily.rename(columns={"time": "observation_date"})
daily["observation_date"] = pd.to_datetime(daily["observation_date"]).dt.date
daily = daily.astype(DAILY_TYPES)
daily = daily.drop_duplicates(subset=["location_id", "observation_date", "data_type"])

df_spark = spark.createDataFrame(daily)
df_spark.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver_weather.weather_daily")